# AgriNav -- Phase-2 Detector Training on curated RICE (Colab / GPU)

**Phase 2.** Fine-tunes the `weeddet_v6b` detector on the **curated RICE** dataset,
warm-started from the **phase-1 RiceSEG backbone**. This notebook only *drives*
`agrinav.training.weeddet_train` from the repo -- it defines no model, loss, or
training logic of its own.

**This is a TWO-class task:** `rice_protect` (COCO id 1), `weed_target` (COCO id 2).
The 3-class RiceSEG detector config does **not** apply; we use
`configs/training/detector_rice_phase2.yaml`.

**Which data, and why this split.** Images/annotations come from the *curated* RICE
build (`agrinav_intake_2026-07-21/deliverable/detection/RICE/`): a grouped,
leakage-free split (capture-series families, 40-frame contiguous blocks,
weed-balanced 70/20/10), with per-image filter decisions and classes already remapped
to the project taxonomy. **Do not substitute the Roboflow-native split inside
`RICE.coco.zip`** -- its train/valid/test share video-frame families (adjacent frames
land on both sides), so any val metric computed on it is inflated.

**Still exploratory.** The deliverable is **loss convergence + qualitative val
predictions**, NOT a defensible mAP. The COCO evaluator exists (`agrinav evaluate`),
but the model-runner adapter and the canonical decode (Gate-4 P1-6) are not done, so
no mAP cell is included yet -- by design, rather than producing a number that would
change once the decode is unified.

**The test split is sealed** and is not even uploaded: `RICE_curated_phase2.zip`
contains only `train` and `valid`.

**Before you run:** `Runtime -> Change runtime type -> GPU`, and in Drive:
- `MyDrive/agrinav_data/rice_phase2/RICE_curated_phase2.zip` -- images + annotations
- `MyDrive/agrinav_data/out/riceseg_backbone.pth` -- the phase-1 backbone


## 1. Confirm GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))
print('torch:', torch.__version__)

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Get the code (clone the private repo)

**This repository is PRIVATE**, so an anonymous clone fails with
`could not read Username for 'https://github.com'` -- Colab has no interactive prompt.

Pick ONE of:

1. **Colab secret (recommended).** Left sidebar -> Secrets -> add `GITHUB_TOKEN` with a
   fine-grained GitHub PAT (**Contents: Read**) and toggle notebook access on. The token
   is never printed, never saved into the notebook, and is stripped from the git remote
   after cloning.
2. **Make the repo public** -- then this cell works with no token.
3. **Skip GitHub:** upload the repo folder to Drive and set `REPO_DIR` to it.

The cell **stops immediately** if the code isn't available, so later cells can't cascade.


In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/Bmerrysmith/Autonomous-tractor-system.git'
BRANCH   = 'master'
REPO_DIR = '/content/agrinav'

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
print('GitHub token found in Colab secrets:', bool(token))

def _redact(s):
    return s.replace(token, '***') if token else s

# GIT_TERMINAL_PROMPT=0 turns an auth failure into an immediate error, not a hang.
env = dict(os.environ, GIT_TERMINAL_PROMPT='0')
auth_url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL

if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git', 'clone', '--branch', BRANCH, auth_url, REPO_DIR],
                       env=env, capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(
            'CLONE FAILED -- stopping so later cells do not cascade.\n\n'
            + _redact(r.stderr) +
            '\nFix: add a GITHUB_TOKEN Colab secret (fine-grained PAT, Contents:Read), '
            'or make the repo public, or upload the repo to Drive and set REPO_DIR.')
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', REPO_URL], env=env)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], env=env)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], env=env)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], env=env)

os.chdir(REPO_DIR)
assert os.path.exists('src/agrinav/training/weeddet_train.py'), (
    f'Repo at {REPO_DIR} has no src/agrinav/training/weeddet_train.py -- wrong branch?')

# The --riceseg-backbone flag landed 2026-07-23. Fail loudly on a stale clone rather
# than silently training from ImageNet and invalidating the phase-1 comparison.
with open('src/agrinav/training/weeddet_train.py', encoding='utf-8') as fh:
    assert 'riceseg-backbone' in fh.read(), (
        'This clone predates --riceseg-backbone. Delete /content/agrinav and re-run.')
print('repo ready at', REPO_DIR)
get_ipython().system('git log --oneline -1')

## 4. Install the package

Colab already ships a CUDA torch/torchvision satisfying the `train` extra's range, so pip
keeps it and only adds the rest.

In [ ]:
get_ipython().system('pip install -q -e ".[train]"')

## 5. Extract the curated RICE data + locate the backbone

Extracts `RICE_curated_phase2.zip` to fast local Colab disk (`/content/rice_curated`),
**not** Drive -- training reads thousands of small files and the Drive mount is far too
slow for that.

The archive holds `images/{train,valid}/` and `annotations/instances_{train,valid}.coco.json`.
**It contains no `test`**, so the sealed split cannot be touched even by accident; the cell
asserts that rather than trusting it.

In [ ]:
import glob, zipfile, json

DRIVE_P2 = '/content/drive/MyDrive/agrinav_data/rice_phase2'
LOCAL    = '/content/rice_curated'

zip_path = f'{DRIVE_P2}/RICE_curated_phase2.zip'
if not os.path.exists(zip_path):
    hits = glob.glob('/content/drive/MyDrive/**/RICE_curated_phase2.zip', recursive=True)
    assert hits, ('RICE_curated_phase2.zip not found in Drive. Upload it to '
                  f'{DRIVE_P2}/ (built locally from the curated RICE deliverable).')
    zip_path = hits[0]
print('zip:', zip_path)

if not os.path.isdir(f'{LOCAL}/images/train'):
    os.makedirs(LOCAL, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        members = z.namelist()
        assert not any('/test/' in m for m in members), (
            'This archive contains a test split -- it must not. Rebuild it train+valid only.')
        z.extractall(LOCAL)

TRAIN_JSON = f'{LOCAL}/annotations/instances_train.coco.json'
VAL_JSON   = f'{LOCAL}/annotations/instances_valid.coco.json'
TRAIN_IMGS = f'{LOCAL}/images/train'
VAL_IMGS   = f'{LOCAL}/images/valid'
for p in (TRAIN_JSON, VAL_JSON, TRAIN_IMGS, VAL_IMGS):
    assert os.path.exists(p), f'missing after extract: {p}'

for name, j in (('train', TRAIN_JSON), ('valid', VAL_JSON)):
    d = json.load(open(j))
    cats = {c['id']: c['name'] for c in d['categories']}
    print(f'{name}: {len(d["images"])} imgs  {len(d["annotations"])} anns  cats={cats}')

BACKBONE = '/content/drive/MyDrive/agrinav_data/out/riceseg_backbone.pth'
assert os.path.exists(BACKBONE), (
    f'phase-1 backbone not found at {BACKBONE}. Run the RiceSEG pretraining notebook '
    'first, or set BACKBONE=None to warm-start from ImageNet instead.')
print('backbone:', BACKBONE)

## 6. Sanity gate: self-test

One forward+backward on synthetic tensors (including a zero-GT image); no data, no network.
Must print PASS before spending GPU time.

In [ ]:
get_ipython().system('python -B -u -m agrinav.training.weeddet_train --self-test')

## 7. Wiring gate: overfit 8 real RICE images

Cheap insurance that the *data* path is right, not just the model path. If the detector
cannot drive the loss down on 8 images it certainly will not learn on 1798 -- and you find
out in a minute rather than an hour. Exits non-zero if the loss fails to decrease.

In [ ]:
cmd = (f'python -B -u -m agrinav.training.weeddet_train '
       f'--ann-file "{TRAIN_JSON}" --images-root "{TRAIN_IMGS}" '
       f'--class-names rice_protect,weed_target '
       f'--overfit 8 --batch-size 2 --img-size 512 --no-pretrained-backbone')
get_ipython().system(cmd)

## 8. Train (exploratory, RiceSEG-warm-started)

Uses `configs/training/detector_rice_phase2.yaml` (2-class, 512px, batch 8, AMP, ~18
epochs) with `--riceseg-backbone`, which loads the phase-1 backbone **instead of**
ImageNet. That loader **fails closed**: every expected `backbone.*` tensor must match by
name and shape or it raises -- a silent partial load cannot happen.

`weeddet_best.pth` is selected by lowest **train** loss; there is no val-based selection
yet (that needs the evaluator adapter), so treat the exported checkpoint as exploratory.

In [ ]:
import datetime, hashlib

TS      = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = f'{DRIVE_P2}/runs/weeddet_rice_{TS}'
os.makedirs(RUN_DIR, exist_ok=True)

CONFIG      = 'configs/training/detector_rice_phase2.yaml'
CLASS_NAMES = 'rice_protect,weed_target'
SEED        = 42

def _git(*a):
    try:
        return subprocess.check_output(['git', '-C', '.', *a], text=True).strip()
    except Exception:
        return None

def _sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

manifest = {
    'run_id': f'weeddet_rice_{TS}',
    'created_utc': datetime.datetime.utcnow().isoformat() + 'Z',
    'phase': 'phase-2 detector, curated RICE, RiceSEG-warm-started',
    'git_commit': _git('rev-parse', 'HEAD'),
    'git_branch': _git('rev-parse', '--abbrev-ref', 'HEAD'),
    'git_dirty': bool(_git('status', '--porcelain')),
    'config_file': CONFIG,
    'class_map': {n: i for i, n in enumerate(CLASS_NAMES.split(','))},
    'seed': SEED,
    'train_split': TRAIN_JSON,
    'val_split': VAL_JSON,
    'dataset': 'curated RICE (grouped 40-frame capture-family split, weed-balanced 70/20/10)',
    'backbone_init': BACKBONE,
    'backbone_sha256': _sha256(BACKBONE),
    'torch': torch.__version__,
    'gpu': torch.cuda.get_device_name(0),
    'note': ('Exploratory: loss convergence + qualitative val predictions only. No mAP '
             '(needs Gate-4 P1-6 canonical decode + evaluator adapter). Test split sealed '
             'and absent from the archive.'),
}
with open(f'{RUN_DIR}/run_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2, sort_keys=True)
print('run dir:', RUN_DIR)
print(json.dumps(manifest, indent=2))

In [ ]:
cmd = (f'python -B -u -m agrinav.training.weeddet_train '
       f'--ann-file "{TRAIN_JSON}" --images-root "{TRAIN_IMGS}" '
       f'--config "{CONFIG}" --riceseg-backbone "{BACKBONE}" '
       f'--seed {SEED} --checkpoint-dir "{RUN_DIR}"')
get_ipython().system(cmd)

## 9. Qualitative predictions on VALIDATION images

Loads the best (EMA) checkpoint and draws predicted boxes on ~6 **validation** images. This
is the exploratory deliverable, **not** a mAP score. Boxes carry the class **name** and score
(never colour alone). The **test split is never touched** -- it is not even present.

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from agrinav.training.weeddet_train import (
    _CocoSplitDataset, load_checkpoint_model, predict_image)

device      = 'cuda' if torch.cuda.is_available() else 'cpu'
class_names = CLASS_NAMES.split(',')
ckpt        = f'{RUN_DIR}/weeddet_best.pth'
assert os.path.exists(ckpt), f'no checkpoint at {ckpt} -- did training finish?'
model = load_checkpoint_model(ckpt, num_classes=len(class_names), device=device)

val_ds = _CocoSplitDataset(VAL_JSON, VAL_IMGS, tuple(class_names), img_size=512)
items  = val_ds.items()
random.Random(0).shuffle(items)
picks  = items[:6]

colors = ['lime', 'red']
fig, axes = plt.subplots(2, 3, figsize=(16, 11))
for ax, (img_id, path, W, H) in zip(axes.ravel(), picks):
    img, boxes, scores, labels = predict_image(
        model, path, img_size=512, device=device, score_thr=0.3)
    ax.imshow(img); ax.set_axis_off()
    ax.set_title(f'{os.path.basename(path)[:28]}  ({len(boxes)} dets @0.3)', fontsize=9)
    for (x1, y1, x2, y2), s, l in zip(boxes, scores, labels):
        c = colors[int(l) % len(colors)]
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                       fill=False, edgecolor=c, linewidth=2))
        ax.text(x1, max(y1 - 4, 0), f'{class_names[int(l)]} {s:.2f}', color=c, fontsize=8,
                bbox=dict(facecolor='black', alpha=0.5, pad=1, edgecolor='none'))
plt.suptitle('Phase-2 RICE exploratory predictions on VAL (no mAP yet; test sealed)',
             fontsize=13)
plt.tight_layout(); plt.show()

## Next steps

1. Read the per-epoch `avg_loss` stream -- the goal is a **smoothly decreasing** loss with
   no NaNs, plus qualitatively sensible boxes on VAL.
2. Artifacts land in the Drive run dir: `weeddet_best.pth`, periodic `weeddet_epochN.pth`,
   and `run_manifest.json` (git commit + dirty flag + config + class map + seed +
   **backbone sha256**, so the run is traceable to the exact phase-1 backbone).
3. **The ImageNet-vs-RiceSEG comparison is one flag:** re-run cell 8 *without*
   `--riceseg-backbone` for the ImageNet control, same seed and config. That is the
   experiment phase-1 exists to justify -- but it is only a fair comparison once
   **val-based selection and mAP** exist, since train loss alone cannot rank two backbones.
4. **Out of scope here:** a calibrated operating point, COCO mAP, and the sealed test-set
   evaluation. Those need Gate-4 P1-6 (one canonical decode) plus the model-runner adapter
   feeding `agrinav evaluate`.